# B3 — `X.Oc138k.roles.e4 → Oc`: Model 2 predicts the reagents too

One data transform fixes two defects at once, and both were invisible until the roles of
the molecules on ORD's reactant side were separated (`scripts/build_conditions_roles.py`).

**The interface between the models.** Model 2 has so far trained on inputs averaging 3.46
molecules, because ORD writes down the whole vessel charge. At inference it is handed
Model 1's output, which under USPTO's convention averages 1.67. Training and serving saw
different input distributions. The mismatch appears exactly when the *best* Model 1 is used,
and it stayed hidden while Model 1 was itself ORD-trained. The input here is substrates
only — 1.71 molecules per record, against 1.71 for USPTO-50K.

**The third role.** A reaction needs substrates, conditions, and the stoichiometric species
in between: K2CO3, NaH, EDC, halide sources. Substrates went to Model 1 and conditions to
Model 2; the reagents were predicted by neither. They live in ORD's reactant field, and only
0.7% of them are duplicated in the solvent or catalyst fields, so no other part of the record
covers them. For a chemist reading a proposed route, that is the missing half of the recipe.
`reagents` is now the first target field.

Target: `reagents|solvent|catalyst|temperature`, e.g. `[Na+], [OH-]|C(C)O|?|?`. Reagents come
first because it is the longest field and the least guessable from a prefix. `yield_percent`
stays dropped: of 5,687 test records the four-field model emitted a yield number 61 times and
abstained 5,234 times, though 60.7% of training rows carry one.

**Field population** over the 138,869 training rows: solvent 89.5%, **reagents 79.9%**,
temperature 26.1%, catalyst 15.7%. Reagents is the second best-populated field in the set.

**Success criterion, fixed before the run** so that the decision to report it is not made
after seeing the number: the field counts as working if its strict top-5 reaches 30% on the
records that have one, **and** solvent and catalyst do not drop more than 2 points against
B2. Otherwise it goes into the limitations, not the results.

**Lengths measured, not guessed** (CompoundT5 vocabulary, 4000 rows): target median 29
tokens, p99 207, max 345 — a 256 cap truncates 0.05%. Input median 105, p99 234 — the 256
cap on the source side truncates about 1%, and the evaluator now takes the same value rather
than a hardcoded one.

**Data:** `kuzmenkoiryna/retro-planner-ord-conditions-roles` (138,869 train + 7,714 val +
5,687 leak-free test).

**Base:** set from B2's result — the two bases were compared on identical data, fields and
format precisely so this run has to happen only once.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob, json

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

# The roles split must have travelled with the data, not been re-derived here.
sample = json.loads(open(test_file).readline())
assert "reagents" in sample and "full_reactants_smiles" in sample, "dataset is the pre-roles one"
print("\ninput   :", sample["reactants_smiles"])
print("reagents:", sample["reagents"])
print("as ORD wrote it:", sample["full_reactants_smiles"])

base_model = "sagawa/CompoundT5"   # set from B2
learning_rate = 5e-4
condition_fields = "reagents,solvent,catalyst,temperature_celsius"
output_dir = "/kaggle/working/model2_conditions_roles"
time_budget_minutes = 400

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 8 x 4 accumulation keeps the effective batch at 32 while fitting a 220M model on a
# T4; the script's 32 x 1 default was set for t5-small and OOMs here at 14.55 of 14.56 GiB.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_roles \
    --target-format compact \
    --condition-fields "{condition_fields}" \
    --max-source-length 256 \
    --max-target-length 256 \
    --per-device-train-batch-size 8 \
    --per-device-eval-batch-size 8 \
    --gradient-accumulation-steps 4 \
    --learning-rate {learning_rate} \
    --num-train-epochs 4 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
!grep -E "new character token|Train examples|condition field" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
marker = json.load(open(f"{output_dir}/final/conditions_format.json"))
print("format marker:", marker)
assert marker["fields"] == condition_fields.split(","), "marker disagrees with the requested fields"

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 12)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
best_epoch = min(points, key=lambda p: p[1])[0]
print(f"  best {state.get('best_metric')} at epoch {best_epoch:.2f} of {points[-1][0]:.2f} reached")

In [ ]:
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 8 --device cuda \
    --max-source-length 256 --max-target-length 256 \
    --output "/kaggle/working/B3_conditions_roles_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/B3_conditions_roles_clean_topk.json"))
summary = data["summary"]
print(json.dumps(summary, indent=2))
print("per-record entries kept for a paired test:", len(data["records"]))

# The criterion, applied as written above rather than reinterpreted now.
reagents_ok = (summary.get("reagents_exact_match_top5") or 0) >= 0.30
print(f"\nreagents strict top-5 = {summary.get('reagents_exact_match_top5')} -> "
      f"{'meets' if reagents_ok else 'misses'} the 30% bar set before the run")
for field in ("solvent", "catalyst"):
    print(f"  {field} strict top-5 = {summary.get(field + '_exact_match_top5')} (compare with B2)")